In [0]:
# CELL 1
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType
from functools import reduce
from pyspark.sql import DataFrame

df = spark.read.table("sarkarimitracatalog.sarkaridatabase.schemes_structured")
print(f"✅ Loaded {df.count()} schemes for chunking")

✅ Loaded 3399 schemes for chunking


In [0]:
# CELL 2 — Define chunking logic
CHUNK_SIZE = 300
OVERLAP    = 50

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    if not text or text.strip() == "":
        return []
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += (chunk_size - overlap)
    return chunks

chunk_udf = F.udf(chunk_text, ArrayType(StringType()))

In [0]:
# CELL 3 — Chunk all text columns
TEXT_COLUMNS = ["details", "benefits", "eligibility", "application", "documents"]

chunk_dfs = []
for col_name in TEXT_COLUMNS:
    temp = df.select("slug", "scheme_name", "level", "schemeCategory", col_name) \
        .filter(F.col(col_name).isNotNull()) \
        .withColumn("chunks", chunk_udf(F.col(col_name))) \
        .withColumn("chunk_text", F.explode("chunks")) \
        .withColumn("source_column", F.lit(col_name)) \
        .withColumn("chunk_id", F.concat_ws("_",
            F.col("slug"),
            F.lit(col_name),
            F.monotonically_increasing_id().cast("string"))) \
        .select("chunk_id", "slug", "scheme_name", "source_column",
                "chunk_text", "level", "schemeCategory")
    chunk_dfs.append(temp)

df_chunks = reduce(DataFrame.union, chunk_dfs)
print(f"✅ Total chunks: {df_chunks.count()}")
display(df_chunks.limit(10))

✅ Total chunks: 18739


chunk_id,slug,scheme_name,source_column,chunk_text,level,schemeCategory
sfmcas-glwb_details_0,sfmcas-glwb,Full Medical Checkup Assistance Scheme- Gujarat Labour Welfare Board,details,"The “Full Medical Checkup Assistance Scheme” is implemented by the Gujarat Labour Welfare Board, Labour, Skill Development & Employment Department, Government of Gujarat. The objective of the scheme is to conduct health check-ups for male Shramyogis above 45 years of age and female Shram Yogis above 35 years of age who are working in the organized sector in the state of Gujarat. Only those workers (Shramyogi) who’s Labour Welfare Fund has been paid regularly by their organization/unit for the last one year will be eligible.",State,Health & Wellness
fa_details_1,fa,Funeral Assistance,details,"The scheme “Funeral Assistance” is implemented by the Maharashtra Building and Other Construction Workers Welfare Board (MBOCWW), Labour Department, Government of Maharashtra. Under this scheme, the Board provides financial assistance for funeral expenses to the nominated heir of a registered worker in the event of their death.",State,Social welfare & Empowerment
faabocwwb_details_2,faabocwwb,Funeral Assistance (A.B.O.C.W.W.B),details,"The scheme “Funeral Assistance” was started by the Assam Building and Other Construction Workers Welfare Board (A.B.O.C.W.W.B), Labour Welfare Department, Government of Assam. Under this scheme, financial assistance shall be given to the nominees/dependents of a deceased registered worker, towards funeral expenses.",State,Social welfare & Empowerment
faanbocwwb_details_3,faanbocwwb,Funeral Assistance (ANBOCWWB),details,"The scheme “Funeral Assistance” was started by the Andaman & Nicobar Islands Building and Other Construction Workers Welfare Board (ANBOCWWB), Department of Labour, Employment & Training, Andaman & Nicobar Administration. Under this scheme, financial assistance shall be provided to the nominees or dependents of deceased workers who were registered under the A&N Islands Building and Other Construction Workers Welfare Board, towards their funeral expenses.",State,Social welfare & Empowerment
facbocwwb_details_4,facbocwwb,Funeral Assistance (CBOCWWB),details,"The scheme “Funeral Assistance” was started by the Chandigarh Building and Other Construction Workers Welfare Board (CBOCWWB), Labour Department, Chandigarh. Under this scheme, financial assistance shall be given to the legal heirs, in case of death of registered workers under the CBOCWWB.",State,Social welfare & Empowerment
fa-gbocwwb_details_5,fa-gbocwwb,Funeral Assistance (GBOCWWB),details,"""Funeral Assistance (GBOCWWB)"" is a Welfare Scheme by the Goa Building and Other Construction Workers Welfare Board of the Department of Labour and Employment, Government of Goa. Through this scheme, the nominee/dependant of the deceased worker is provided with financial assistance of ₹5,000/-. The applications are accepted offline.",State,Social welfare & Empowerment
fahbocwwb_details_6,fahbocwwb,Funeral Assistance (HBOCWWB),details,"The scheme “Funeral Assistance” is implemented by the Haryana Building and Other Construction Workers Welfare Board (HBOCWWB), Labour Department, Government of Haryana. Under this scheme, the Board provides financial assistance for funeral expenses to the nominee or legal heir of a registered worker in the event of their death.",State,Social welfare & Empowerment
fasgbocwwb_details_7,fasgbocwwb,Funeral Assistance Scheme (GBOCWWB),details,"The “Funeral Assistance Scheme” is implemented by the Gujarat Building and Other Construction Worker’s Welfare Board (GBOCWWB), Labour, Skill Development & Employment Department, Government of Gujarat. The main objective of this scheme is to provide financial assistance to the heir of a deceased construction worker registered with the Gujarat Building and Other Construction Worker’s Welfare Board in case of death during the ongoing membership.",State,Social welfare & Empowerment
fasmp_details_8,fasmp,Fun

In [0]:
# CELL 3 — Chunk all text columns
TEXT_COLUMNS = ["details", "benefits", "eligibility", "application", "documents"]

chunk_dfs = []
for col_name in TEXT_COLUMNS:
    temp = df.select("slug", "scheme_name", "level", "schemeCategory", col_name) \
        .filter(F.col(col_name).isNotNull()) \
        .withColumn("chunks", chunk_udf(F.col(col_name))) \
        .withColumn("chunk_text", F.explode("chunks")) \
        .withColumn("source_column", F.lit(col_name)) \
        .withColumn("chunk_id", F.concat_ws("_",
            F.col("slug"),
            F.lit(col_name),
            F.monotonically_increasing_id().cast("string"))) \
        .select("chunk_id", "slug", "scheme_name", "source_column",
                "chunk_text", "level", "schemeCategory")
    chunk_dfs.append(temp)

df_chunks = reduce(DataFrame.union, chunk_dfs)
print(f"✅ Total chunks: {df_chunks.count()}")
display(df_chunks.limit(10))

✅ Total chunks: 18739


chunk_id,slug,scheme_name,source_column,chunk_text,level,schemeCategory
sfmcas-glwb_details_0,sfmcas-glwb,Full Medical Checkup Assistance Scheme- Gujarat Labour Welfare Board,details,"The “Full Medical Checkup Assistance Scheme” is implemented by the Gujarat Labour Welfare Board, Labour, Skill Development & Employment Department, Government of Gujarat. The objective of the scheme is to conduct health check-ups for male Shramyogis above 45 years of age and female Shram Yogis above 35 years of age who are working in the organized sector in the state of Gujarat. Only those workers (Shramyogi) who’s Labour Welfare Fund has been paid regularly by their organization/unit for the last one year will be eligible.",State,Health & Wellness
fa_details_1,fa,Funeral Assistance,details,"The scheme “Funeral Assistance” is implemented by the Maharashtra Building and Other Construction Workers Welfare Board (MBOCWW), Labour Department, Government of Maharashtra. Under this scheme, the Board provides financial assistance for funeral expenses to the nominated heir of a registered worker in the event of their death.",State,Social welfare & Empowerment
faabocwwb_details_2,faabocwwb,Funeral Assistance (A.B.O.C.W.W.B),details,"The scheme “Funeral Assistance” was started by the Assam Building and Other Construction Workers Welfare Board (A.B.O.C.W.W.B), Labour Welfare Department, Government of Assam. Under this scheme, financial assistance shall be given to the nominees/dependents of a deceased registered worker, towards funeral expenses.",State,Social welfare & Empowerment
faanbocwwb_details_3,faanbocwwb,Funeral Assistance (ANBOCWWB),details,"The scheme “Funeral Assistance” was started by the Andaman & Nicobar Islands Building and Other Construction Workers Welfare Board (ANBOCWWB), Department of Labour, Employment & Training, Andaman & Nicobar Administration. Under this scheme, financial assistance shall be provided to the nominees or dependents of deceased workers who were registered under the A&N Islands Building and Other Construction Workers Welfare Board, towards their funeral expenses.",State,Social welfare & Empowerment
facbocwwb_details_4,facbocwwb,Funeral Assistance (CBOCWWB),details,"The scheme “Funeral Assistance” was started by the Chandigarh Building and Other Construction Workers Welfare Board (CBOCWWB), Labour Department, Chandigarh. Under this scheme, financial assistance shall be given to the legal heirs, in case of death of registered workers under the CBOCWWB.",State,Social welfare & Empowerment
fa-gbocwwb_details_5,fa-gbocwwb,Funeral Assistance (GBOCWWB),details,"""Funeral Assistance (GBOCWWB)"" is a Welfare Scheme by the Goa Building and Other Construction Workers Welfare Board of the Department of Labour and Employment, Government of Goa. Through this scheme, the nominee/dependant of the deceased worker is provided with financial assistance of ₹5,000/-. The applications are accepted offline.",State,Social welfare & Empowerment
fahbocwwb_details_6,fahbocwwb,Funeral Assistance (HBOCWWB),details,"The scheme “Funeral Assistance” is implemented by the Haryana Building and Other Construction Workers Welfare Board (HBOCWWB), Labour Department, Government of Haryana. Under this scheme, the Board provides financial assistance for funeral expenses to the nominee or legal heir of a registered worker in the event of their death.",State,Social welfare & Empowerment
fasgbocwwb_details_7,fasgbocwwb,Funeral Assistance Scheme (GBOCWWB),details,"The “Funeral Assistance Scheme” is implemented by the Gujarat Building and Other Construction Worker’s Welfare Board (GBOCWWB), Labour, Skill Development & Employment Department, Government of Gujarat. The main objective of this scheme is to provide financial assistance to the heir of a deceased construction worker registered with the Gujarat Building and Other Construction Worker’s Welfare Board in case of death during the ongoing membership.",State,Social welfare & Empowerment
fasmp_details_8,fasmp,Fun

In [0]:
# CELL 4 — Write
df_chunks.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sarkarimitracatalog.sarkaridatabase.scheme_chunks")

print("✅ Written: sarkarimitracatalog.sarkaridatabase.scheme_chunks")

✅ Written: sarkarimitracatalog.sarkaridatabase.scheme_chunks
